## Code for Training a Digit Image Recognition Model

In [ ]:
# 0. Import necessary libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

print("PyTorch Version:", torch.__version__)
print("Torchvision Version:", torchvision.__version__)

# 1. Data Preparation Phase
print("Starting data preparation...")

# Convert images to tensors and normalize to a range between -1 and 1.
# The SVHN dataset also uses 3-channel color images, so we use the same preprocessing method as CIFAR-10.
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

# Load the training dataset.
# root: data storage path, split: 'train' or 'test', download: whether to download automatically
trainset = torchvision.datasets.SVHN(root='./data', split='train',
                                     download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=4,
                                          shuffle=True, num_workers=2)

# Load the test dataset.
testset = torchvision.datasets.SVHN(root='./data', split='test',
                                    download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=4,
                                         shuffle=False, num_workers=2)

# Define class names (numbers from 0 to 9)
classes = ('0', '1', '2', '3', '4', '5', '6', '7', '8', '9')


# Function for data visualization
def imshow(img):
    img = img / 2 + 0.5     # Unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

# Check some of the training data
dataiter = iter(trainloader)
images, labels = next(dataiter)

# Print images and labels
imshow(torchvision.utils.make_grid(images))
print(' '.join(f'{classes[labels[j]]:5s}' for j in range(4)))
print("Data preparation complete!")
print("-" * 30)

# 2. Define the CNN Model
# Since the input image size (3x32x32) is the same as CIFAR-10, we use the same model structure.
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10) # Final output is 10 (numbers 0-9)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Create model object and allocate to GPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
net = Net().to(device)
print("Model definition complete!")
print(net)
print("-" * 30)


# 3. Define the Loss Function and Optimizer
# We use CrossEntropyLoss and SGD as before for multi-class classification.
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)
print("Loss function and optimizer definition complete!")
print("-" * 30)


# 4. Model Training
print("Starting model training...")
for epoch in range(3):  # Train for only 3 epochs due to time constraints.

    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # Move data and labels to the device
        inputs, labels = data[0].to(device), data[1].to(device)

        # Initialize the optimizer's gradients
        optimizer.zero_grad()

        # Forward propagation -> Calculate loss -> Backpropagation -> Update parameters
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # Print loss value every 5000 mini-batches
        running_loss += loss.item()
        if i % 5000 == 4999:
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 5000:.3f}')
            running_loss = 0.0

print("Model training complete!")
# Save the trained model
PATH = './svhn_net.pth'
torch.save(net.state_dict(), PATH)
print(f"Trained model saved to {PATH}")
print("-" * 30)


# 5. Model Evaluation
print("Starting model evaluation...")
correct = 0
total = 0
# No need to calculate gradients during evaluation
with torch.no_grad():
    for data in testloader:
        images, labels = data[0].to(device), data[1].to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the model on the 26032 test images: {100 * correct // total} %')

# Check accuracy by class
class_correct = list(0. for i in range(10))
class_total = list(0. for i in range(10))
with torch.no_grad():
    for data in testloader:
        images, labels = data[0].to(device), data[1].to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)
        c = (predicted == labels).squeeze()
        # Exception handling for cases where the batch size is not 4
        if len(labels) == 4:
            for i in range(4):
                label = labels[i]
                class_correct[label] += c[i].item()
                class_total[label] += 1

for i in range(10):
    # Prevent division by zero
    if class_total[i] > 0:
        print(f'Accuracy for class {classes[i]} : {100 * class_correct[i] / class_total[i]:.1f} %')
    else:
        print(f'Class {classes[i]} is not in the test data.')

print("Model evaluation complete!")

## Code for Using the Digit Image Recognition Model

In [ ]:
# 0. Import necessary libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from google.colab import files
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

# 1. Define the model architecture (the model's skeleton is needed to load the saved weights)
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# 2. Load the trained model
# Set the GPU or CPU device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Create model object and move to device
model = Net().to(device)

# Path to the saved model weights (state_dict)
PATH = './svhn_net.pth'

# Load weights
model.load_state_dict(torch.load(PATH))

# Set the model to 'evaluation mode' (disables Dropout, BatchNorm, etc.)
model.eval()
print("Successfully loaded the trained SVHN model.")
print("-" * 30)


# 3. Define the user image preprocessing pipeline
# Transform the user-uploaded image to match the model's input specifications (3x32x32).
transform_user_image = transforms.Compose([
    transforms.Resize((32, 32)),  # 1. Force resize the image to 32x32
    transforms.ToTensor(),         # 2. Convert to PyTorch tensor
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # 3. Normalize the same way as during training
])


# 4. Image upload and prediction function
def predict_digit(model, transform):
    # Use Colab's file upload feature
    uploaded = files.upload()

    # Get the name of the uploaded file
    for fn in uploaded.keys():
        print(f"Analyzing image '{fn}'.")

        # Open the image
        image = Image.open(fn).convert('RGB') # Convert other formats like RGBA to RGB

        # Show the original image
        plt.imshow(image)
        plt.title("Original image uploaded by user")
        plt.axis('off')
        plt.show()

        # Preprocess the image to match the model input
        image_tensor = transform(image)

        # The model expects input in the form (batch size, channel, height, width), so
        # add a 'batch' dimension at the beginning. (C, H, W) -> (1, C, H, W)
        image_tensor = image_tensor.unsqueeze(0).to(device)

        # Disable gradient calculation
        with torch.no_grad():
            # Input the image into the model to get the results (outputs)
            outputs = model(image_tensor)

            # Select the index of the class with the highest score as the prediction
            _, predicted = torch.max(outputs, 1)

        # Define class names
        classes = ('0', '1', '2', '3', '4', '5', '6', '7', '8', '9')

        # Print the prediction result
        predicted_class = classes[predicted.item()]
        print("\n" + "="*30)
        print(f"🤖 Model prediction: ✨ {predicted_class} ✨")
        print("="*30)

# 5. Execute the function
# When you run the function below, a file upload